In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
sys.path.insert(0, str(Path.cwd().parent / 'python'))
from analysis_common import analytical_src, segment_metrics
from config import AGGREGATE_DIR, AUDIT_DIR, CLEAN_PARQUET, DOCS_DIR, STATS_DIR, VISUALS_DIR, connect
pd.set_option('display.max_columns', 40, 'display.width', 180, 'display.float_format', '{:,.4f}'.format)
con = connect(); src = analytical_src()
kpi = json.loads((AGGREGATE_DIR / 'portfolio_kpis.json').read_text())
fin = json.loads((AGGREGATE_DIR / 'financial_totals.json').read_text())
show = lambda name: display(Image(filename=str(VISUALS_DIR / f'{name}.png')))
csv = lambda name: pd.read_csv(AGGREGATE_DIR / f'{name}.csv')

# 2. Data Cleaning
Pipeline: `python/data_cleaning.py`, checked by `python/data_validation.py`.

In [2]:
json.loads((AUDIT_DIR / 'cleaning_counts.json').read_text())

{'rows_out': 2260668,
 'distinct_ids': 2260668,
 'max_last_pymnt': '2019-03-01',
 'dti_placeholder': 137,
 'dti_missing': 1711,
 'dti_gt100': 2426,
 'income_zero': 1667,
 'income_gt1m': 583,
 'util_100_150': 7315,
 'util_gt150': 28,
 'util_missing': 1802,
 'negative_late_fee': 8,
 'min_late_fee': -9.499999999999999e-09,
 'il999': 2,
 'rev999': 1,
 'home_other': 1232,
 'emp_missing': 146907,
 'inv_zero': 233,
 'unmapped_status': 0,
 'raw_rows': 2260701,
 'footer_rows': 33}

In [3]:
v = pd.read_csv(AUDIT_DIR / 'validation_results.csv'); print(v.severity.value_counts().to_dict(), 'failed errors:', ((~v.passed) & (v.severity=='error')).sum()); v

{'error': 31, 'info': 6} failed errors: 0


,rule,measured,passed,severity
0,"Staging rows == 2,260,701 documented source re...","2,260,701.0000",True,error
1,Clean rows == raw rows - 33 footers,"2,260,668.0000",True,error
2,Loan id unique,"2,260,668.0000",True,error
3,Loan id never null,0.0000,True,error
4,No value lost in type conversion (non-null cou...,0.0000,True,error
5,SUM(loan_amnt) unchanged by cleaning,"34,016,115,925.0000",True,error
6,SUM(funded_amnt) unchanged by cleaning,"34,004,208,600.0000",True,error
7,SUM(total_pymnt) unchanged by cleaning,"27,314,649,582.6200",True,error
8,SUM(total_rec_prncp) unchanged by cleaning,"21,489,393,643.5800",True,error
9,SUM(recoveries) unchanged by cleaning,"325,262,956.8600",True,error


Live check: original values stay untouched while cleaned fields sit beside them.

In [4]:
con.execute(f"SELECT dti_flag, COUNT(*) n, MIN(dti) min_raw, MAX(dti) max_raw, MAX(dti_clean) max_clean FROM {src} GROUP BY 1 ORDER BY n DESC").df()

,dti_flag,n,min_raw,max_raw,max_clean
0,ok,2256394,0.0000,100.0000,100.0000
1,suspicious_gt100,2426,100.0400,995.6000,995.6000
2,missing,1711,NaN,NaN,NaN
3,placeholder,137,-1.0000,999.0000,NaN


In [5]:
con.execute(f"SELECT annual_inc_flag, COUNT(*) n, MIN(annual_inc) min_raw, MAX(annual_inc) max_raw FROM {src} GROUP BY 1 ORDER BY n DESC").df()

,annual_inc_flag,n,min_raw,max_raw
0,ok,2258414,0.3600,"1,000,000.0000"
1,zero,1667,0.0000,0.0000
2,suspicious_gt1m,583,"1,000,001.0000","110,000,000.0000"
3,missing,4,NaN,NaN
